# Data Science for Business - Subset selection on Ames Housing

*Session 5 · ISLP 6.1.1–6.1.3 (Algorithm 6.2)*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
np.random.seed(42)
plt.style.use('fivethirtyeight')

## Problem description

Ask a home buyer to describe their dream house, and they probably won't begin with the height of the basement ceiling or the proximity to an east-west railroad. But this dataset proves that much more influences price negotiations than the number of bedrooms or a white-picket fence. With 76 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa, this dataset challenges you to predict the final price of each home. More: <https://www.kaggle.com/c/house-prices-advanced-regression-techniques>

## Load data

Load data from CSV file.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_05/ameshousing.csv')

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data.columns

## Prepare data

First, we will remove some columns that are not useful for our task.

In [ ]:
data = data.drop(['YrSold', 'MoSold', 'SaleCondition', 'SaleType'], axis=1)

Next, we will split the data into predictors (*X*) and response (*y*) and into training (*X_train, y_train*) and test (*X_test, y_test*) sets.

In [ ]:
X = data.drop("SalePrice", axis=1)
y = data["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Finally, we will do some feature engineering. It is important to use only information from the training set for feature engineering, and then mechanistically repeat these steps on the test set.

Typically, feature engineering depends strongly on the datatype of the variables. Hence, we will first determine which variables are categorical and which are numerical. Subsequently, we will transform these variables separately.

In [ ]:
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns
numerical_features = X_train.select_dtypes(exclude=['object', 'string']).columns

In [ ]:
categorical_features

In [ ]:
numerical_features

The categorical variables must be transformed into numerical representations, e.g., by one-hot encoding them.

In [ ]:
enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
enc.fit(X_train[categorical_features])

X_train_cat = enc.transform(X_train[categorical_features])
X_test_cat = enc.transform(X_test[categorical_features])

X_train_cat = pd.DataFrame(X_train_cat, columns=enc.get_feature_names_out(categorical_features))
X_test_cat = pd.DataFrame(X_test_cat, columns=enc.get_feature_names_out(categorical_features))

In [ ]:
X_train_cat.head()

The numerical variables will be standardized, that is, we will subtract the mean and divide by the standard deviation. The scaler is fit on the training data only.

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train[numerical_features])

X_train_num = scaler.transform(X_train[numerical_features])
X_test_num = scaler.transform(X_test[numerical_features])

X_train_num = pd.DataFrame(X_train_num, columns=numerical_features)
X_test_num = pd.DataFrame(X_test_num, columns=numerical_features)

In [ ]:
X_train_num.head()

Join categorical and numerical features. We standardize only the numerical features; the dummies are already 0/1.

In [ ]:
X_train = pd.concat([X_train_num, X_train_cat], axis=1)
X_test = pd.concat([X_test_num, X_test_cat], axis=1)

In [ ]:
X_train.head()

In [ ]:
X_train.shape

## Stepwise (forward/backward) feature selection

*Best subset selection* (ISLP 6.1.1, Algorithm 6.1) would fit all $2^p$ possible models. With $p \approx 290$ predictors this is impossible (the book puts the practical limit at around 40 predictors). Stepwise selection (ISLP 6.1.2) is a greedy, computationally cheap alternative.

### Feature selection with pre-defined number of features

Forward stepwise selection (ISLP 6.1.2, Algorithm 6.2) starts with no predictors and greedily adds the predictor that improves the fit most. We use sklearn's `SequentialFeatureSelector`, which can do *forward* and *backward* selection.

Note a small difference to the book: sklearn evaluates each candidate within a step by 5-fold CV RMSE, whereas Algorithm 6.2 uses the training RSS (or $R^2$) within each step and uses Cp/BIC/adjusted $R^2$ or CV only to choose the number of predictors (ISLP 6.1.3).

In [ ]:
lm = LinearRegression()
sfs_fwd = SequentialFeatureSelector(lm, n_features_to_select=3, direction='forward', scoring='neg_root_mean_squared_error', cv=5)
sfs_fwd.fit(X_train, y_train)

Show the selected features. Note: The features are NOT listed in the order of their importance, but in the order they appear in the dataset!

In [ ]:
sfs_fwd.get_feature_names_out()

To see how well a model with these selected features performs, we train it on the full training data and evaluate it on the test data. CV on the training data chose the features; the test set is only used for this final evaluation.

In [ ]:
mod_selected_features = LinearRegression().fit(X_train[sfs_fwd.get_feature_names_out()], y_train)
preds_selected_features = mod_selected_features.predict(X_test[sfs_fwd.get_feature_names_out()])
print(root_mean_squared_error(y_test, preds_selected_features))

### Choosing the number of features by cross-validation

So far, we simply decided to use 3 features. How many should we use? ISLP 6.1.3 recommends estimating the test error of the best model of each size $k$ by cross-validation **on the training data** and picking the size with the lowest CV error.

The loop below runs forward selection for $k = 1, \dots, 10$, computes the 5-fold CV RMSE of each selected model, and stores the results in a list. Forward selection is nested: each step keeps the features of the previous step and adds one (ISLP Algorithm 6.2). We therefore also record which feature was added in each step, so the `features` column lists the features in the order they were added. This takes about a minute.

In [ ]:
results = []
features_so_far = []

for k in range(1, 11):
    sfs = SequentialFeatureSelector(lm, n_features_to_select=k, direction='forward', scoring='neg_root_mean_squared_error', cv=5)
    sfs.fit(X_train, y_train)
    selected = sfs.get_feature_names_out()

    added = [f for f in selected if f not in features_so_far]
    features_so_far = features_so_far + added

    cv_rmse = -cross_val_score(lm, X_train[features_so_far], y_train, cv=5, scoring='neg_root_mean_squared_error')

    results.append({'n_features': k,
                    'cv_rmse': cv_rmse.mean(),
                    'added_feature': added[0],
                    'features': features_so_far})

In [ ]:
results_df = pd.DataFrame(results)
results_df

Plot the CV RMSE against the number of features (cf. ISLP Fig. 6.3).

In [ ]:
plt.plot(results_df['n_features'], results_df['cv_rmse'], marker='o')
plt.xlabel('Number of features')
plt.ylabel('CV RMSE')
plt.show()

The CV error drops quickly for the first few features and then decreases only slowly: each additional feature helps less and less, while the model gets more complex (bias-variance trade-off).

We choose the number of features with the lowest CV RMSE. The minimum is at the edge of our range, so more features would likely help further; we stop at 10 to keep the runtime short.

In [ ]:
k_chosen = results_df.loc[results_df['cv_rmse'].idxmin(), 'n_features']
print('Lowest CV RMSE with', k_chosen, 'features')

Fit the chosen model on the full training data and evaluate it once on the test set.

In [ ]:
features_chosen = results_df.loc[results_df['n_features'] == k_chosen, 'features'].iloc[0]
print(features_chosen)

mod_chosen = LinearRegression().fit(X_train[features_chosen], y_train)
preds_chosen = mod_chosen.predict(X_test[features_chosen])
print(root_mean_squared_error(y_test, preds_chosen))

## Your turn!

**Your turn!** *Fewer vs. all features.* Fit a linear regression on **all** features of `X_train` and compare its test RMSE with the forward-selected model above. Which model is better, and why?

In [ ]:
# YOUR CODE HERE

**Your turn!** *Business sense check.* Look at the selected features in the order they were added (`results_df`). Do they make sense for house prices? Add one feature you would expect to matter but that wasn't selected, refit the model, and compare the test RMSE. (Hint: `X_train.columns` lists all available features.)

In [ ]:
# YOUR CODE HERE